<a href="https://colab.research.google.com/github/VasilinaFrolova/MyPain/blob/main/%D0%A4%D1%80%D0%BE%D0%BB%D0%BE%D0%B2%D0%B0_%D0%92%D0%B0%D1%81%D0%B8%D0%BB%D0%B8%D0%BD%D0%B0_%22fine_tuning_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [ ]:
# Устанавливаем необходимые библиотеки
!pip install transformers datasets evaluate accelerate

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
from datasets import load_dataset
import evaluate
import numpy as np

# 1. Загружаем датасет AG News
dataset = load_dataset("ag_news")
print(dataset)

# 2. Загружаем токенизатор и модель (distilbert-base-uncased)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4  # 4 класса: World, Sports, Business, Sci/Tech
)

# 3. Токенизируем данные
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Преобразуем в формат, ожидаемый Trainer (убираем ненужные колонки)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Определяем метрику accuracy
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# 5. Настраиваем параметры обучения
training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=500,
    report_to="none",                   # отключаем wandb, если не нужен
)

# 6. Создаем Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# 7. Обучаем модель
trainer.train()

# 8. Оцениваем на тестовой выборке
eval_results = trainer.evaluate()
print(f"Test accuracy: {eval_results['eval_accuracy']:.4f}")

# 9. Сохраняем модель
trainer.save_model("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")
print("Модель сохранена в папку ./ag_news_model")

# 10. Тестируем на трех новых новостях с помощью pipeline
classifier = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model",
    device=0 if torch.cuda.is_available() else -1
)

# Примеры новостей
test_news = [
    "Iraq cleric 'to end Najaf revolt' Shia cleric Moqtada Sadr reportedly agrees to end an uprising in the holy Iraqi city of Najaf.",
    "Greek sprinters quit Games ATHENS (Reuters) - Greece #39;s top two sprinters have quit the Olympic Games after submitting their country to six days of embarrassment in a hide-and-seek contest with anti-doping enforcers.",
    "Microsoft finalises three-year government deal Hot on the heels of its 10-year strategic partnership with the London Borough of Newham, Microsoft is close to signing a new broad three-year public sector agreement with the government."
]

# Соответствие меток классам AG News (индексы: 0-World, 1-Sports, 2-Business, 3-Sci/Tech)
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

for news in test_news:
    result = classifier(news)[0]
    label_id = int(result['label'].split('_')[-1])  # на случай, если метка вида 'LABEL_1'
    # В некоторых версиях pipeline возвращает 'label': 'LABEL_0', поэтому преобразуем
    # Можно также использовать id2label из модели, но проще так:
    predicted_class = id2label[label_id]
    confidence = result['score']
    print(f"Текст: {news}")
    print(f"Предсказанный класс: {predicted_class}, уверенность: {confidence:.4f}\n")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

...

Test accuracy: 0.9491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в папку ./ag_news_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Текст: Iraq cleric 'to end Najaf revolt' Shia cleric Moqtada Sadr reportedly agrees to end an uprising in the holy Iraqi city of Najaf.
Предсказанный класс: World, уверенность: 0.9994

Текст: Greek sprinters quit Games ATHENS (Reuters) - Greece #39;s top two sprinters have quit the Olympic Games after submitting their country to six days of embarrassment in a hide-and-seek contest with anti-doping enforcers.
Предсказанный класс: Sports, уверенность: 0.9997

Текст: Microsoft finalises three-year government deal Hot on the heels of its 10-year strategic partnership with the London Borough of Newham, Microsoft is close to signing a new broad three-year public sector agreement with the government.
Предсказанный класс: Sci/Tech, уверенность: 0.5935

